# Introduction
The purpose of this project is to find the most optimal time combinations for each race.
This will be determined by weather conditionns, lap times per driver, tire compound and age, sector times, speed traps, and pit stop data.

# Sources
https://www.redbull.com/us-en/how-does-weather-impact-f1

# Tires Explanation

## 1. The Soft Tire
The softest compound selected for the weekend, designed for absolute maximum speed over a short period.

### Pros
* **Highest Grip:** Provides the highest level of mechanical grip, allowing cars to corner and brake much faster.
* **Instant Warm-up:** Reaches its ideal operating temperature almost immediately (often within just a few corners).
* **Qualifying King:** The definitive tire for setting the fastest possible lap time when fuel loads are low.

### Cons
* **High Thermal Degradation:** Overheats very quickly, especially on high-energy tracks or in hot weather.
* **Short Lifespan:** The physical rubber wears away rapidly; it can rarely sustain a long race stint without lap times dropping off heavily.

## 2. The Medium Tire 
The "all-rounder" tire that acts as the baseline compromise between raw speed and endurance.

### Pros
* **Great Versatility:** Provides a highly competitive pace while offering a much longer lifespan than the Softs.
* **The Race Favorite:** Usually forms the backbone of a race strategy, allowing teams flexibility to extend a stint if necessary.
* **Balanced Warm-up:** Takes slightly longer to warm up than a Soft, but handles high fuel loads at the start of a race much better.

### Cons
* **Master of None:** It is slower than the Soft tire over a single lap, and doesn't last as long as the Hard tire.

## 3. The Hard Tire 
The sturdiest slick compound selected for the weekend, engineered for maximum durability.

### Pros
* **Extreme Longevity:** Can withstand dozens of laps under heavy fuel loads without losing its structural integrity.
* **Thermal Stability:** Highly resistant to overheating, making it excellent for high-track-temperature environments or high-speed tracks with demanding corners (like Silverstone or Spa).
* **Strategic Predictability:** Delivers very consistent, flat lap times for the majority of its life.

### Cons
* **Lowest Grip:** Lacks the peak cornering and traction capabilities of the Soft and Medium.
* **Difficult Warm-up:** Takes one to three full push-laps to reach its ideal operating window. Drivers are highly vulnerable to being overtaken right after exiting the pit lane on cold Hard tires.

## 4. The Intermediate Tire
The go-to tire for damp, changing, or unpredictable weather conditions.

### Pros
* **Clears Standing Water:** Features grooves (treads) designed to displace water from the track surface to prevent the car from floating.
* **Crossover Agility:** Functions across a wide window, coping well when a track is drying out or when light rain is actively falling.

### Cons
* **Disintegrates on Dry Asphalt:** Without water to cool the soft tread blocks, the tire will overheat, melt, and lose all grip within a few laps on a dry track.

## 5. The Full Wet Tire 
The heavy-duty tire designed for extreme rainfall and heavily saturated track conditions.

### Pros
* **Massive Water Displacement:** Capable of clearing incredible amounts of water per second at high speed, minimizing the risk of dangerous aquaplaning.
* **Safety in Torrential Rain:** Keeps the car controllable even when there are rivers running across the asphalt.

### Cons
* **High Rolling Resistance:** The deep tread pattern and heavy structure make the car significantly slower overall.
* **Very Narrow Window:** If the rain eases even slightly and the track transitions to just being "damp," the Full Wet rapidly loses performance compared to an Intermediate and destroys its own tread.


In [31]:
# Imports
import fastf1 as f1
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# Weather Analysis

# Variable Choice & Significance

#### Weather
Significant Conditions

* Rain 
* Snow
* Ice
* Fog
* Thunder & Lightning
* Wind

#### Track Temperature
In F1 the hotter the track is the less grip there is on the track. This is becuase the tires are preheated and driving around the track generates heat. This leads to tires overheating and wear out faster. Conversly, if the track is cold then the tires require more energy to get going and the tires lose pressure quicker. 

#### Humidity
Moisture on the track reduces grip and traction.

#### Rainfall
When it rains, the track goes from being a dry surface to a wet one. This forces teams to move away from "slick" tires (Soft, Medium, Hard) to treaded tires (Intermediates or Full Wets).

In [32]:
#List of session information
session = f1.get_session(2024,1,'R') #Gets the session info, Year, Race, Type of Race
session.load() #Load session data
gp = session.event['EventName']

#Calcualtions
avg_airtemps = session.weather_data['TrackTemp'].mean()
mean_error_airtemps = session.weather_data['TrackTemp'].sem()
avg_humidity = session.weather_data['Humidity'].mean()
mean_error_humidity = session.weather_data['Humidity'].sem()
rain = session.weather_data['Rainfall'].any()


#Print Information
print("----------------------------------")
print("Race Informationa & Calculations")
print(session.name) #What kind of session it is
print(session.date) #When the session was
print(gp) #What race
print(f"The average track tempreatures for {gp} are {avg_airtemps:.2f} + or - {mean_error_airtemps:.2f}")
print(f"The average humidity for {gp} is {avg_humidity:.2f} + or - {mean_error_humidity:.2f}")
if(rain == True):
    print(f"It did rain during the {gp}")
else:
    print(f"It did not rain during the {gp}")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


----------------------------------
Race Informationa & Calculations
Race
2024-03-02 15:00:00
Bahrain Grand Prix
The average track tempreatures for Bahrain Grand Prix are 23.65 + or - 0.11
The average humidity for Bahrain Grand Prix is 48.82 + or - 0.14
It did not rain during the Bahrain Grand Prix


# Lap and Tire Analysis

In [33]:
laps = session.laps #Get lap information
drivers = session.drivers #Get driver information

cols = ['Driver', 'LapNumber', 'Stint', 'LapTime', 'Compound', 'TyreLife']
laps_clean = laps[cols].copy()
laps_clean['LapTimeSeconds'] = laps_clean['LapTime'].dt.total_seconds()

#Filter out dirty laps
laps_clean = laps_clean.dropna(subset=['LapTimeSeconds', 'Compound'])
laps_clean = laps_clean[laps_clean['LapTimeSeconds'] > 0]

# Remove obvious outliers (laps way slower than typical race pace)
threshold = laps_clean['LapTimeSeconds'].quantile(0.95)
laps_clean = laps_clean[laps_clean['LapTimeSeconds'] < threshold]

print("----------------------------------")
print(laps_clean['Compound'].value_counts())
print(laps_clean['Driver'].unique())
print("----------------------------------")
print(laps_clean.head(20))

----------------------------------
Compound
HARD    754
SOFT    316
Name: count, dtype: int64
['VER' 'PER' 'SAI' 'LEC' 'RUS' 'NOR' 'HAM' 'PIA' 'ALO' 'STR' 'ZHO' 'MAG'
 'RIC' 'TSU' 'ALB' 'HUL' 'OCO' 'GAS' 'BOT' 'SAR']
----------------------------------
   Driver  LapNumber  Stint                LapTime Compound  TyreLife  \
0     VER        1.0    1.0 0 days 00:01:37.284000     SOFT       4.0   
1     VER        2.0    1.0 0 days 00:01:36.296000     SOFT       5.0   
2     VER        3.0    1.0 0 days 00:01:36.753000     SOFT       6.0   
3     VER        4.0    1.0 0 days 00:01:36.647000     SOFT       7.0   
4     VER        5.0    1.0 0 days 00:01:37.173000     SOFT       8.0   
5     VER        6.0    1.0 0 days 00:01:37.092000     SOFT       9.0   
6     VER        7.0    1.0 0 days 00:01:37.038000     SOFT      10.0   
7     VER        8.0    1.0 0 days 00:01:37.024000     SOFT      11.0   
8     VER        9.0    1.0 0 days 00:01:37.229000     SOFT      12.0   
9     VER       10

In [34]:
#Get median lap time
median_lap_time = laps_clean['LapTimeSeconds'].median()
mean_error_lapTime = laps_clean['LapTimeSeconds'].sem()
print(f"The median lap time is {median_lap_time:.2f} + or - {mean_error_lapTime:.2f}")

#Calcuate Degredation Rate using Linear Regression
# compounds = laps_clean['Compound'].unique()

# for compound in compounds:
#     subset = laps_clean[laps_clean['Compound'] == compound]
    
#     x = subset['TyreLife'].values
#     y = subset['LapTimeSeconds'].values
    
#     slope, intercept = np.polyfit(x, y, 1)
    
#     print(f"{compound}: {slope:.4f} seconds/lap  (base pace: {intercept:.3f}s)")

#Calcuate Degradation Rate by Driver using Linear Regression
grouped_drivers = laps_clean.groupby(['Driver','Stint'])
results = []

for (driver,stint), group in grouped_drivers:

    x = group['TyreLife'].values
    y = group['LapTimeSeconds'].values

    slope, intercept = np.polyfit(x, y, 1)

    results.append({
        'Driver': driver,
        'Stint': stint,
        'Compound': group['Compound'].iloc[0],
        'DegRate': slope,
        'BasePace': intercept
    })


The median lap time is 96.95 + or - 0.04


In [35]:
#Datafram of Tire Degredation per driver
df = pd.DataFrame(results)
print(df)

mean_degRate = df.groupby('Compound')['DegRate'].mean()
print(mean_degRate)

   Driver  Stint Compound   DegRate   BasePace
0     ALB    1.0     SOFT  0.095547  98.103134
1     ALB    2.0     HARD  0.052375  97.472635
2     ALB    3.0     HARD  0.065202  95.569833
3     ALO    1.0     SOFT  0.005346  98.847523
4     ALO    2.0     HARD  0.046549  96.765711
..    ...    ...      ...       ...        ...
57    VER    2.0     HARD  0.109384  94.569037
58    VER    3.0     SOFT  0.033532  94.315942
59    ZHO    1.0     SOFT  0.342619  97.394095
60    ZHO    2.0     HARD  0.106743  96.709865
61    ZHO    3.0     HARD  0.039571  96.262586

[62 rows x 5 columns]
Compound
HARD    0.065874
SOFT    0.116096
Name: DegRate, dtype: float64


# Generating an Effciency Score

In [36]:
#Integral to get score: beta0*n + 1/2*beta1*n^2

u = laps_clean.groupby(['Driver','Stint']).size()

stint_lengths = u.reset_index()
stint_lengths.columns = ['Driver', 'Stint', 'StintLength']
print(stint_lengths.head())

df = df.merge(stint_lengths, on=['Driver', 'Stint'])

avg_stint = df['StintLength'].mean()

def eff_function(row):
    beta0 = row['BasePace']
    beta1 = row['DegRate']
    n = avg_stint
    result = (beta0*n) + (0.5)*(beta1*(n**2))
    return result

df['EfficiencyScore']= df.apply(eff_function,axis=1)
print(df.head())

  Driver  Stint  StintLength
0    ALB    1.0           14
1    ALB    2.0           20
2    ALB    3.0           19
3    ALO    1.0           15
4    ALO    2.0           25
  Driver  Stint Compound   DegRate   BasePace  StintLength  EfficiencyScore
0    ALB    1.0     SOFT  0.095547  98.103134           14      1707.299152
1    ALB    2.0     HARD  0.052375  97.472635           20      1689.988763
2    ALB    3.0     HARD  0.065202  95.569833           19      1659.060221
3    ALO    1.0     SOFT  0.005346  98.847523           15      1706.713115
4    ALO    2.0     HARD  0.046549  96.765711           25      1676.921009


In [37]:
eff_score_per_compound = df.groupby('Compound')['EfficiencyScore'].mean()
print(eff_score_per_compound)

Compound
HARD    1669.776413
SOFT    1691.212441
Name: EfficiencyScore, dtype: float64


In [ ]:
print('---------------------')
print(mean_degRate)
print('---------------------')
print(eff_score_per_compound)


---------------------
Compound
HARD    0.065874
SOFT    0.116096
Name: DegRate, dtype: float64
---------------------
Compound
HARD    1669.776413
SOFT    1691.212441
Name: EfficiencyScore, dtype: float64
